# C12-classical-models — Practice p30 — Solution


All custom runs use the same deterministic repair and objective contract. Selection, co-clustering, and optimal center permutation are computed as separate auditable objects.


In [ ]:
import itertools
import numpy as np
from sklearn.cluster import KMeans

X_p30 = np.array([[0.,0.],[0.,0.],[0.,2.],[1.,1.],
                  [8.,8.],[8.,10.],[9.,9.],[9.,9.],
                  [4.,4.],[4.,5.]], dtype=np.float64)
initializations_p30 = [
    np.array([[0.,0.],[0.,0.],[9.,9.]], dtype=np.float64),
    np.array([[0.,2.],[4.,4.],[8.,10.]], dtype=np.float64),
    np.array([[1.,1.],[4.,5.],[9.,9.]], dtype=np.float64),
]


def robust_lloyd(X, initial_centroids, max_iter=100, tol=1e-10):
    if not all(isinstance(a,np.ndarray) and a.dtype==np.float64 for a in (X,initial_centroids)):
        raise ValueError("float64 arrays required")
    if X.ndim!=2 or initial_centroids.ndim!=2 or X.shape[0]<1 or X.shape[1]<1:
        raise ValueError("invalid dimensions")
    if initial_centroids.shape[1]!=X.shape[1] or not 1<=initial_centroids.shape[0]<=X.shape[0]:
        raise ValueError("invalid centroid shape")
    if not np.isfinite(X).all() or not np.isfinite(initial_centroids).all(): raise ValueError("finite values required")
    if not isinstance(max_iter,(int,np.integer)) or isinstance(max_iter,bool) or max_iter<=0: raise ValueError("positive max_iter required")
    if not np.isscalar(tol) or not np.isfinite(tol) or tol<0: raise ValueError("nonnegative tol required")
    centroids=initial_centroids.copy(); previous=None; trace=[]; repair_count=0; k=len(centroids)
    for n_iter in range(1,int(max_iter)+1):
        distances=((X[:,None,:]-centroids[None,:,:])**2).sum(axis=2)
        labels=np.argmin(distances,axis=1).astype(np.int64)
        assignment=distances[np.arange(len(X)),labels].copy()
        for empty in np.flatnonzero(np.bincount(labels,minlength=k)==0):
            counts=np.bincount(labels,minlength=k); eligible=np.flatnonzero(counts[labels]>=2)
            farthest=assignment[eligible].max()
            row=int(eligible[np.flatnonzero(assignment[eligible]==farthest)[0]])
            labels[row]=int(empty); repair_count+=1
        updated=np.vstack([X[labels==cluster].mean(axis=0) for cluster in range(k)]).astype(np.float64)
        trace.append(float(((X-updated[labels])**2).sum()))
        unchanged=previous is not None and np.array_equal(labels,previous)
        movement=float(np.linalg.norm(updated-centroids,axis=1).max()); centroids=updated
        if unchanged or movement<=tol: break
        previous=labels.copy()
    return {"labels":labels,"centroids":centroids,"objective_trace":np.asarray(trace,dtype=np.float64),
            "n_iter":int(n_iter),"repair_count":int(repair_count)}


runs_p30 = tuple(robust_lloyd(X_p30, init) for init in initializations_p30)
selected_index_p30 = int(np.argmin([run["objective_trace"][-1] for run in runs_p30]))
co_clustering_p30 = np.stack([run["labels"][:,None]==run["labels"][None,:] for run in runs_p30]).astype(bool)
upper_p30=np.triu_indices(len(X_p30),k=1)
pairwise_agreements_p30=np.empty((3,3),dtype=np.float64)
for i_p30 in range(3):
    for j_p30 in range(3):
        pairwise_agreements_p30[i_p30,j_p30]=np.mean(co_clustering_p30[i_p30][upper_p30]==co_clustering_p30[j_p30][upper_p30])
sklearn_model_p30=KMeans(n_clusters=3,init="k-means++",n_init=10,max_iter=100,tol=1e-10,algorithm="lloyd",random_state=20260804).fit(X_p30)
sklearn_inertia_p30=float(sklearn_model_p30.inertia_)
sklearn_centers_p30=sklearn_model_p30.cluster_centers_.astype(np.float64,copy=True)
selected_centers_p30=runs_p30[selected_index_p30]["centroids"]
permutations_p30=tuple(itertools.permutations(range(3)))
matching_costs_p30=np.array([np.sum((selected_centers_p30-sklearn_centers_p30[list(permutation)])**2) for permutation in permutations_p30])
center_matching_p30=np.array(permutations_p30[int(np.argmin(matching_costs_p30))],dtype=np.int64)
comparison_p30='''The custom WCSS traces are nonincreasing and every repaired cluster is nonempty. The selected custom run and sklearn both attain WCSS 7 here. Co-clustering, not raw ids, compares partitions. The lexicographically first minimum-cost center permutation aligns centers up to label permutation; equality here is observed rather than assumed.'''


### Answer check


In [ ]:
ATOL=1e-10
RTOL=1e-8
assert all(set(run)=={"labels","centroids","objective_trace","n_iter","repair_count"} for run in runs_p30)
assert selected_index_p30==0
assert np.allclose([run["objective_trace"][-1] for run in runs_p30],[7.0,7.0,7.0],atol=ATOL,rtol=RTOL)
assert co_clustering_p30.shape==(3,10,10) and co_clustering_p30.dtype==bool
assert pairwise_agreements_p30.shape==(3,3) and pairwise_agreements_p30.dtype==np.float64
assert np.allclose(pairwise_agreements_p30,np.ones((3,3)),atol=ATOL,rtol=RTOL)
assert np.isclose(sklearn_inertia_p30,7.0,atol=ATOL,rtol=RTOL)
assert center_matching_p30.shape==(3,) and center_matching_p30.dtype==np.int64
assert np.allclose(selected_centers_p30,sklearn_centers_p30[center_matching_p30],atol=ATOL,rtol=RTOL)
for run_p30 in runs_p30:
    assert run_p30["n_iter"]==len(run_p30["objective_trace"])
    assert np.all(np.bincount(run_p30["labels"],minlength=3)>0)
    assert np.all(np.diff(run_p30["objective_trace"])<=ATOL+RTOL*np.abs(run_p30["objective_trace"][:-1]))
assert "raw ids" in comparison_p30
